In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
tqdm.pandas()

from sklearn.metrics import classification_report, confusion_matrix

import sys
sys.path.append('../code/poseEvaluation/')

from rag import search

from typing import List, Annotated
from pydantic import BaseModel, Field
from enum import Enum

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from dotenv import load_dotenv
load_dotenv()

True

### 1. 예상 질의
* 예상 질의는 90개로 구성
* 3개의 intent로 구분
    1. 복합 질의
        * 다단계의 논리적 처리 또는 다중 도구 호출이 필요한 질의(40개)
            * RAG + Text to SQL
            * 복잡한 Text to SQL
        * 하나의 도구 호출로 완료될 수 없으며, 중간 결과의 통합, 비교, 추론을 위해 plan and excute 구조 기반 답변
        * 예시
            * 나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점이 무엇이야?
            * 내가 올린 영상 중 평균 점수 80점 미만인 영상들의 가장 흔한 오류 부위는 무엇이야?
            * 내가 올린 영상 중 평균 점수 95점 이상인 영상들의 공통적인 특징을 분석하고, 기술 구조를 참조하여 요약해줘.
    2. 단순 질의
        * 하나의 도구 호출이 필요한 질의(30개)
            * 단순 RAG
            * 단순 Text to SQL
        * 예시
            * 지면 반발력(Ground Reaction Force)이 역도 동작에 어떻게 활용되나요?
            * 스내치(Snatch) 동작에서 바벨을 받는 시점에 대한 기술적 조언을 해주세요.
            * 가장 최근에 올린 영상의 전체 점수는 몇점이야?
    3. 예외 질의
        * 서비스의 범위를 벗어나거나(일반 대화, 날씨 등), 현재 지원되지 않는 기능(영상 편집, 계정 관리 등)에 대한 질의(20개)
        * 기존 저장된 메시지로 답변 출력력
        * 예시
            * 파이썬으로 DTW 알고리즘을 구현하는 코드를 알려주세요.
            * 가장 가까운 역도 체육관 위치를 찾아주세요.
            * 영상 ID 500을 삭제해 주세요.

In [11]:
query_path = '..\\data\\rag\\intent_queries_90.csv'

query_df = pd.read_csv(query_path)

In [5]:
test_df = query_df.groupby('category').sample(frac=0.2)
query_df['type'] = np.nan
query_df.loc[~query_df.index.isin(test_df.index), 'type'] = 'validation'
query_df.loc[query_df.index.isin(test_df.index), 'type'] = 'test'

# query_df.to_csv(query_path, encoding='utf-8-sig', index=False)

C:\Users\owner\AppData\Local\Temp\ipykernel_25824\4281981954.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'validation' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  query_df.loc[~query_df.index.isin(test_df.index), 'type'] = 'validation'


### 2. State

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

from langgraph.graph import START, END, StateGraph

In [8]:
from typing import Literal

class State(TypedDict):
    intent: Annotated[Literal["COMPLEX", "SIMPLE", "INAPPROPRIATE"], "Intent Category"]
    messages: Annotated[list, add_messages]

In [8]:
graph_builder = StateGraph(State)



### 3. Intent Analyze
#### 개요
* 사용자 쿼리의 intent(의도) 분석 및 query rewrite 작업
* intent 종류:
    1. COMPLEX: 다단계 처리 또는 멀티툴 사용 요구 질의 (plan & execute sub-graph 처리)
    2. SIMPLE: 단일 응답으로 해결 가능한 간단 질의
    3. INAPPROPRIATE: 서비스 범위 외 또는 부적절 질의

#### 실험 결과
* intent 분석용 프롬프트 기법 비교(COT, COD)

* COT 기법 성능
    * 전체 정확도: 80.0%
    * 전체 Precision: 83.53%, Recall: 80.00%, F1-Score: 76.22%
    * COMPLEX: Precision 81.63%, Recall 100.00%, F1-Score 89.89%
    * SIMPLE: Precision 100.00%, Recall 40.00%, F1-Score 57.14%
    * INAPPROPRIATE: Precision 68.97%, Recall 100.00%, F1-Score 81.63%
    * 특징: COMPLEX와 INAPPROPRIATE는 Recall 100%로 완벽히 포착하나, Precision이 낮아 false positive 발생
    * SIMPLE의 Recall 40%로 SIMPLE 쿼리의 대부분을 COMPLEX로 오분류하는 경향

* COD 기법 성능
    * 전체 정확도: 80.0%
    * 전체 Precision: 86.80%, Recall: 73.06%, F1-Score: 72.45%
    * COMPLEX: Precision 97.37%, Recall 92.50%, F1-Score 94.87%
    * SIMPLE: Precision 63.04%, Recall 96.67%, F1-Score 76.32%
    * INAPPROPRIATE: Precision 100.00%, Recall 30.00%, F1-Score 46.15%
    * 특징: COMPLEX와 SIMPLE에서 균형잡힌 성능, INAPPROPRIATE의 Recall 30%로 오분류 위험

* 분석 및 결론
    * 두 기법 모두 전체 정확도 80.0%로 동일
    * COT: INAPPROPRIATE Recall 100%로 부적절 쿼리를 확실히 차단하지만, SIMPLE 쿼리의 대부분을 COMPLEX로 오분류
    * COD: COMPLEX와 SIMPLE에서 더 균형잡힌 성능을 보이지만, INAPPROPRIATE Recall 30%로 부적절 쿼리를 놓칠 위험 높음
    * 서비스 관점: INAPPROPRIATE 분류 실패는 사용자 경험에 직접적 영향 → COT 기법 선택 권장
    * COT의 SIMPLE→COMPLEX 오분류는 서비스 관점에서 큰 문제 아님 (더 많은 처리 리소스 사용하지만 결과는 제공)

In [4]:
llm = ChatOpenAI(
    temperature=0.2,
    model="gpt-4o-mini",
)

In [ ]:
class Category(str, Enum):
    """Defines the query processing categories."""
    COMPLEX = "COMPLEX"
    SIMPLE = "SIMPLE"
    INAPPROPRIATE = "INAPPROPRIATE"

class Intent(BaseModel):
    """Schema containing the user query’s intent, processing category, and rewritten query."""
    category: Category = Field(
        description="Query processing category. Must be one of 'COMPLEX', 'SIMPLE', or 'INAPPROPRIATE'."
    )
    query_rewrite: str = Field(
        description="A rewritten version of the original query to make it easier for downstream Agents to process. If the category is INAPPROPRIATE, contains a rejection message."
    )

parser = PydanticOutputParser(pydantic_object=Intent)

#### 3-01. COT

In [ ]:
# from langchain.prompts import PromptTemplate

intent_template = """
[SYSTEM ROLE]
You are the Intent Router for a weightlifting analysis service. Your task is to analyze the user's query and determine the appropriate processing Category and a Rewritten Query for the next processing step.

[CATEGORY DEFINITIONS]
1. COMPLEX: Requires multi-step processing, multi-tool usage (SQL + RAG/LLM Inference), or logical comparison/analysis (Plan-and-Execute).
2. SIMPLE: Requires a single tool call (RAG or Single SQL) (Execute Only).
3. INAPPROPRIATE: Outside the service scope (OUT_OF_SCOPE) or unsupported feature (UNHANDLED_TOOL).

[OUTPUT FORMAT]
{format}

[FEW-SHOT EXAMPLE]

User Query: "What are the common chronic faults in my Snatch and Clean & Jerk movements?"

Thought:
1. Analysis: The user is asking for a comparison and synthesis of data from two different lift types (Snatch and C&J).
2. Tooling: This requires multiple SQL queries (Snatch data, C&J data), followed by LLM inference to find commonalities, and finally RAG for technical explanation.
3. Conclusion: This is a multi-step process requiring planning. The Category is COMPLEX.

Output:
{{
  "category": "COMPLEX",
  "query_rewrite": "Calculate the overall average DTW score of all user videos and compare it with the DTW score of the most recently uploaded video."
}}

[USER QUERY]
{user_query}
"""

In [ ]:
prompt = PromptTemplate.from_template(template=intent_template)
prompt = prompt.partial(format=parser.get_format_instructions())
cot_chain = prompt | llm

In [ ]:
query_df['cot_category'] = np.nan
query_df['cot_rewrite'] = np.nan

cot_category_list = []
cot_rewrite_list = []

for i, row in tqdm(query_df.iterrows()):
    # result = cot_chain.invoke({'user_query': row['query']})
    structed_output = parser.parse(result.content)
    cot_category_list.append(structed_output.category.value)
    cot_rewrite_list.append(structed_output.query_rewrite)

query_df['cot_category'] = cot_category_list
query_df['cot_rewrite'] = cot_rewrite_list

# query_df.to_csv('..\\data\\rag\\intent_queries_90.csv', encoding='utf-8-sig', index=False)

In [65]:
query_df.head(2)

,category,query,type,cot_category,cot_rewrite
0,COMPLEX,어제 올린 용상 영상과 저번 주에 올린 영상 중 가장 잘했던 영상의 DTW 분석 결...,validation,COMPLEX,Compare the DTW analysis results of the best v...
1,COMPLEX,나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점이 무엇이야?,validation,COMPLEX,나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점을 분석해줘.


In [14]:
# 전체 정확도
cot_accuracy = round(len(query_df.loc[(query_df['category'] == query_df['cot_category'])]) / len(query_df) * 100, 2)
print(f"COT 전체 정확도: {cot_accuracy}%")

# 분류 보고서 (Precision, Recall, F1-Score 포함)
cot_report = classification_report(
    query_df['category'], 
    query_df['cot_category'],
    target_names=['COMPLEX', 'INAPPROPRIATE', 'SIMPLE'],
    output_dict=True
)

print("\n=== COT 분류 보고서 ===")
print(f"전체 Precision: {cot_report['macro avg']['precision']:.2%}")
print(f"전체 Recall: {cot_report['macro avg']['recall']:.2%}")
print(f"전체 F1-Score: {cot_report['macro avg']['f1-score']:.2%}")

print("\n=== 카테고리별 성능 ===")
for category in ['COMPLEX', 'INAPPROPRIATE', 'SIMPLE']:
    if category in cot_report:
        print(f"\n{category}:")
        print(f"  Precision: {cot_report[category]['precision']:.2%}")
        print(f"  Recall: {cot_report[category]['recall']:.2%}")
        print(f"  F1-Score: {cot_report[category]['f1-score']:.2%}")
        print(f"  Support: {cot_report[category]['support']}")

COT 전체 정확도: 80.0%

=== COT 분류 보고서 ===
전체 Precision: 83.53%
전체 Recall: 80.00%
전체 F1-Score: 76.22%

=== 카테고리별 성능 ===

COMPLEX:
  Precision: 81.63%
  Recall: 100.00%
  F1-Score: 89.89%
  Support: 40.0

INAPPROPRIATE:
  Precision: 68.97%
  Recall: 100.00%
  F1-Score: 81.63%
  Support: 20.0

SIMPLE:
  Precision: 100.00%
  Recall: 40.00%
  F1-Score: 57.14%
  Support: 30.0


#### 3-02. COD

In [72]:
intent_template = """
[SYSTEM ROLE]
You are the Intent Router and Query Rewriter. Analyze the user's query and output the appropriate Category and a rewritten version of the query. 
Use the Chain-of-Draft (CoD) reasoning format.

[CATEGORY DEFINITIONS]
1. COMPLEX: Requires multi-step reasoning or multiple tools (e.g., SQL + RAG).
2. SIMPLE: Can be handled with a single tool (e.g., RAG or single SQL query).
3. INAPPROPRIATE: Out of service scope or unsupported feature.

[OUTPUT FORMAT]
{format}

[FEW-SHOT EXAMPLE]
User Query: "Compare Snatch and Clean & Jerk techniques to find common faults."

Draft:
- Requires comparison of multiple datasets.
- Multi-step reasoning → COMPLEX.

Output:
{{
  "category": "COMPLEX",
  "query_rewrite": "Identify and compare technical faults between Snatch and Clean & Jerk movements."
}}

User Query: "Why should the bar stay close to the body during a Snatch pull?"

Draft:
- Knowledge question.
- Single RAG retrieval → SIMPLE.

Output:
{{
  "category": "SIMPLE",
  "query_rewrite": "Explain the technical reason for keeping the bar close to the body during the Snatch pull."
}}

[USER QUERY]
{user_query}
"""

In [73]:
prompt = PromptTemplate.from_template(template=intent_template)
prompt = prompt.partial(format=parser.get_format_instructions())
cod_chain = prompt | llm

In [ ]:
query_df['cod_category'] = np.nan
query_df['cod_rewrite'] = np.nan

cod_category_list = []
cod_rewrite_list = []

for i, row in tqdm(query_df.iterrows()):
    # result = cod_chain.invoke({'user_query': row['query']})
    structed_output = parser.parse(result.content)
    cod_category_list.append(structed_output.category.value)
    cod_rewrite_list.append(structed_output.query_rewrite)

query_df['cod_category'] = cod_category_list
query_df['cod_rewrite'] = cod_rewrite_list

# query_df.to_csv('..\\data\\rag\\intent_queries_90.csv', encoding='utf-8-sig', index=False)

90it [04:01,  2.68s/it]


In [15]:
cod_accuracy = round(len(query_df.loc[(query_df['category'] == query_df['cod_category'])]) / len(query_df) * 100, 2)
print(f"COD 전체 정확도: {cod_accuracy}%")

# 분류 보고서 (Precision, Recall, F1-Score 포함)
cod_report = classification_report(
    query_df['category'], 
    query_df['cod_category'],
    target_names=['COMPLEX', 'INAPPROPRIATE', 'SIMPLE'],
    output_dict=True
)

print("\n=== COD 분류 보고서 ===")
print(f"전체 Precision: {cod_report['macro avg']['precision']:.2%}")
print(f"전체 Recall: {cod_report['macro avg']['recall']:.2%}")
print(f"전체 F1-Score: {cod_report['macro avg']['f1-score']:.2%}")

print("\n=== 카테고리별 성능 ===")
for category in ['COMPLEX', 'INAPPROPRIATE', 'SIMPLE']:
    if category in cod_report:
        print(f"\n{category}:")
        print(f"  Precision: {cod_report[category]['precision']:.2%}")
        print(f"  Recall: {cod_report[category]['recall']:.2%}")
        print(f"  F1-Score: {cod_report[category]['f1-score']:.2%}")
        print(f"  Support: {cod_report[category]['support']}")

COD 전체 정확도: 80.0%

=== COD 분류 보고서 ===
전체 Precision: 86.80%
전체 Recall: 73.06%
전체 F1-Score: 72.45%

=== 카테고리별 성능 ===

COMPLEX:
  Precision: 97.37%
  Recall: 92.50%
  F1-Score: 94.87%
  Support: 40.0

INAPPROPRIATE:
  Precision: 100.00%
  Recall: 30.00%
  F1-Score: 46.15%
  Support: 20.0

SIMPLE:
  Precision: 63.04%
  Recall: 96.67%
  F1-Score: 76.32%
  Support: 30.0


### 4. Text to SQL
* 결과가 없을 때, humman in the loop
* 기존에 db 함수 연결하기
* 표준 용어사전 적용(body-part)
* user_id를 통해 user가 가져올 수 있는 정보의 범위 제한

In [ ]:
class SqlState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: Annotated[str, "User ID"]
    

In [ ]:
sqlWorkflow = StateGraph(SqlState)

sqlWorkflow.add_node("first_call", first_call)


#### 4-01. 쿼리 재작성 프롬프트 보고서

- 목적: 자연어로 입력된 사용자의 질의를 DB에서 쉽게 조회할 수 있는 쿼리로 프롬프트 기반 변환한다.
- 프롬프트의 주요 기능 및 규칙은 다음과 같다.
    1. Temporal Standardization (시간 표준화)
        - "어제", "지난주" 등 상대적인 시간 표현을 절대적 날짜나 기간으로 바꿔준다.
        - 예시: "어제" → "2025-11-05" (현재 날짜 기준).
    2. Exercise Standardization (동작명 표준화)
        - 사용자 표현이 표준 동작명이 아닐 경우, Exercise Map을 참조해 표준화한다.
        - 예시: "클린 엔 저크", "clean and jerk" → "인상".
    3. Body Part Standardization & Grouping (신체부위 명시 및 묶음)
        - 사용자가 "하체", "상체" 등 추상적 그룹을 언급하면 Keypoint List의 구체 부위로 치환하여 명시한다.
        - 예시: "하체" → "왼쪽 엉덩이", "오른쪽 엉덩이", "왼쪽 무릎", "오른쪽 무릎", "왼쪽 발목", "오른쪽 발목".
    4. Clarity (명확성)
        - 최종 쿼리는 Text-to-SQL Agent에 전달될 수 있도록 명확하고, 문맥이 충분하며 애매성이 없어야 한다.
- 프롬프트에는 현재 시각, Exercise Map, Keypoint List 등이 컨텍스트로 포함되어 있음.
- 최종 출력 쿼리는 DB 질의 단계에서 쉽게 사용할 수 있도록 구성된다.

- 예시
    - 입력: "어제 영상에서 하체 부위의 점수는 몇 점이야?"
    - 처리 흐름:
        1. "어제" → "2025-10-01"
        2. "하체" → ['왼쪽 엉덩이','오른쪽 엉덩이','왼쪽 무릎','오른쪽 무릎','왼쪽 발목','오른쪽 발목']
        3. 재작성 결과:  "2025-10-01 영상에서 '왼쪽 엉덩이', '오른쪽 엉덩이', '왼쪽 무릎', '오른쪽 무릎', '왼쪽 발목', '오른쪽 발목' 부위의 평균 점수를 조회."

In [2]:
class QueryRewriteSQL(BaseModel):
    """ Schema for the rewritten user query to enhance downstream DB querying. """
    rewritten_query: str = Field(
        ...,
        description="A string that rewrites the natural language query into a DB-friendly format."
    )

parser = PydanticOutputParser(pydantic_object=QueryRewriteSQL)

In [ ]:
rewriteSql_template = """
[SYSTEM ROLE]
You are the Query Rewriter Agent. Your sole task is to rewrite the user's natural language query into a clear, unambiguous, and DB-friendly format. This rewritten query will be passed directly to the Text-to-SQL Agent.

[REWRITING RULES]
1. Temporal Standardization: Resolve all relative time expressions (e.g., "어제", "저번 주") into absolute date ranges or specific dates using the provided Current Datetime.
2. Exercise Standardization: Replace all non-standard exercise names with the standardized Korean names from the Exercise Map.
3. Body Part Standardization & Grouping:
    *   If the user mentions an abstract group (e.g., "다리", "하체", "몸통"), you must infer the corresponding list of Keypoint Connection Names (e.g., "왼쪽어깨-왼쪽팔꿈치") and include them in the rewritten query.
    *   The rewritten query must explicitly list the Keypoint Connection Names instead of the abstract group name.
4. Clarity: The final rewritten query must be a clear, self-contained instruction for the Text-to-SQL Agent.

[CONTEXT]
- Current Datetime: {current_datetime}
- Exercise Map: {{
    "클린 엔 저크": "인상",
    "clean and jerk": "인상",
    "스내치": "용상",
    "snatch": "용상"
    }}
- Keypoint Connection Names: [
    "왼쪽어깨-오른쪽어깨", "왼쪽어깨-왼쪽팔꿈치", "왼쪽어깨-왼쪽엉덩이",
     오른쪽어깨-오른쪽팔꿈치", "오른쪽어깨-오른쪽엉덩이",
    "왼쪽팔꿈치-왼쪽손목", "오른쪽팔꿈치-오른쪽손목", 
    "왼쪽엉덩이-오른쪽엉덩이", "왼쪽엉덩이-왼쪽무릎", "오른쪽엉덩이-오른쪽무릎", 
    "왼쪽무릎-왼쪽발목", "오른쪽무릎-오른쪽발목"
    ]

[OUTPUT FORMAT]
{format}

[FEW-SHOT EXAMPLES]

Example 1 (Temporal & Exercise Standardization)
User Query: "오늘 올린 클린 엔 저크 영상의 평균 점수를 알려줘."
Draft (CoT):
1. Temporal: "오늘" -> {current_datetime}.
2. Exercise: "클린 엔 저크" -> "인상".
3. Rewrite: {current_datetime}에 업로드된 인상 영상의 평균 점수를 조회."

Example 2 (Abstract Grouping - 팔)
User Query: "가장 최근 영상에서 팔 점수는 몇 점이야?"
Draft (CoT):
1. Temporal: "가장 최근" -> No change (TtS Agent will handle 'MAX(created_at)').
2. Grouping: "팔" is an abstract group. Infer keypoints: 팔('왼쪽어깨-왼쪽팔꿈치', '왼쪽팔꿈치-왼쪽손목', '오른쪽어깨-오른쪽팔꿈치', '오른쪽팔꿈치-오른쪽손목').
3. Rewrite: "가장 최근 영상에서 '왼쪽어깨-왼쪽팔꿈치', '오른쪽어깨-오른쪽팔꿈치', '왼쪽팔꿈치-왼쪽손목', '오른쪽팔꿈치-오른쪽손목' 부위들의 평균 점수를 조회."

[USER QUERY]
{user_query}
"""

In [6]:
prompt = PromptTemplate.from_template(template=rewriteSql_template)
prompt = prompt.partial(format=parser.get_format_instructions())
rewrite_sql_chain = prompt | llm

result = rewrite_sql_chain.invoke({'user_query': '어제 올린 클린 엔 저크 영상의 분석 결과에서 상체에서 가장 낮은 점수를 조회해줘.', 'current_datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S')})

parser.parse(result.content).rewritten_query

"2025-11-07에 업로드된 인상 영상의 분석 결과에서 '왼쪽어깨-오른쪽어깨', '왼쪽어깨-왼쪽팔꿈치', '오른쪽어깨-오른쪽팔꿈치' 부위들의 가장 낮은 점수를 조회."

In [8]:
result = rewrite_sql_chain.invoke({'user_query': '그제 올린 snatch 분석 결과에서 골반에서 가장 낮은 점수를 조회해줘.', 'current_datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S')})

parser.parse(result.content).rewritten_query

"2025-11-06 01:42:49에 업로드된 용상 분석 결과에서 '왼쪽엉덩이-오른쪽엉덩이' 부위의 가장 낮은 점수를 조회."

In [9]:
result = rewrite_sql_chain.invoke({'user_query': '이번달 초 즈음 분석 결과에서 종아리에서 가장 낮은 점수를 조회해줘.', 'current_datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S')})

parser.parse(result.content).rewritten_query

"2025-11-01 00:00:00부터 2025-11-08 01:45:16 사이의 분석 결과에서 '왼쪽엉덩이-왼쪽무릎', '오른쪽엉덩이-오른쪽무릎', '왼쪽무릎-왼쪽발목', '오른쪽무릎-오른쪽발목' 부위의 가장 낮은 점수를 조회."

In [ ]:
def first_call(state: SqlState) -> dict[str, list[AIMessage]]:
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "sql_db_list_tables",
                        "args": {},
                        "id": "inital_call"
                    }
                ]
            )
        ]
    }

In [ ]:
{
  "tables": {
    "pose_scores": {
      "columns": {
        "id": "int, primary key, auto increment",
        "session_id": "int, foreign key → pose_evaluation_sessions.id",
        "body_part": "varchar(100), not null",
        "average_score": "decimal(5,2), not null",
        "is_below_standard": "tinyint(1), default 0",
      }
    },
    "sport_standard_scores": {
      "columns": {
        "id": "int, primary key, auto increment",
        "video_id": "int, foreign key → videos.id",
        "body_part": "varchar(50), not null",
        "body_part_korean": "varchar(100)",
        "standard_score": "decimal(5,2), not null"
      }
    },
    "pose_evaluation_sessions": {
      "columns": {
        "id": "int, primary key, auto increment",
        "user_id": "int, foreign key (users.id, optional)",
        "sport_id": "int, foreign key → sports.id",
        "user_video_id": "int, foreign key (videos.id, optional)",
        "standard_video_id": "int, foreign key (videos.id, optional)",
        "session_name": "varchar(200)",
        "created_at": "timestamp, default CURRENT_TIMESTAMP"
      }
    },
    "sports": {
      "columns": {
        "id": "int, primary key, auto increment",
        "name": "varchar(100), not null",
        "category": "varchar(50)",
        "description": "text",
      }
    }
  },
  "relationships": [
    {
      "from": "pose_scores.session_id",
      "to": "pose_evaluation_sessions.id"
    },
    {
      "from": "pose_evaluation_sessions.sport_id",
      "to": "sports.id"
    },
    {
      "from": "sport_standard_scores.video_id",
      "to": "videos.id"
    }
  ]
}

* pose_scores, average_score, user_score, sport_standard_scores의 standard_score와 대비하여, 이 점수가 사용자 영상의 점수임을 명확히 합니다.
* pose_scores, body_part, connection_index, body_part가 11-12와 같은 인덱스 문자열을 저장한다는 것을 명확히 하여, LLM이 이 필드를 일반적인 신체 부위 이름으로 오해하는 것을 방지합니다.
* sport_standard_scores, body_part, connection_index, pose_scores와 동일하게 인덱스 문자열임을 명확히 합니다.
* sport_standard_scores, body_part_korean, connection_name, 이 필드가 부위 연결명(왼쪽어깨-오른쪽어깨)을 저장한다는 것을 명확히 합니다.
* pose_evaluation_sessions, user_video_id, user_video_id, videos 테이블과의 관계가 명확하지 않아 LLM이 혼동할 수 있습니다. videos 테이블의 스키마를 추가하거나, 이 필드가 사용자 영상의 ID임을 명확히 해야 합니다.

In [ ]:
from typing import Any

from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks
from langgraph.prebuilt import ToolNode

# 오류 처리와 에이전트에 오류를 전달하는 기능
def handle_tool_error(state) -> dict:
    error = state['error']

    tool_calls = state['messages'][-1].tool_calls

    return {
        "messages": [
            ToolMessage(
                content=f"Here is the error: {repr(error)}\n\nPlease fix your mistakes.",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

def create_tool_node_with_fallback(tools: list) -> RunnableWithFallbacks[Any, dict]:
    """
    Create a ToolNode with a fallback to handle errors and surface them to the agent.
    """
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )

### Tool 
1. search: 역도 체육지도자 훈련지도서 기반 답변(rag)
2. load: pose estimation + dtw 기반 자세 분석 결과
    * 고민중
        * 특정 sql을 사용할 지
        * 아니면 text to sql 기반의 검색결과 로드를 진행할지
    * 입력값
        * user_id(무조건)
        * 
3. hitl: human in the loop
    * load 과정에서 결과 선택에 인간이 개입
    * 필요할 때에만, (이걸 또 생각해야 함.)

### text to sql
* 

In [ ]:
search('clean 동작 중, 팔 동작에 문제 발견')

c:\Users\owner\Desktop\workspace\SportAgent\jupyter\../code/poseEvaluation\rag.py:153: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


['성공적인 상\n담 진행을 위해서 상담사는 내담자의 감정에 공감할 수 있어야 한다. 또한 열정이\n있되 유머를 잃지 않고 내담자를 대하는 동시에, 상담 과정에서 발생하는 문제를 창\n의적으로 해결할 수 있어야 한다. 상담자는 상담의 진행과정에서 상담과 관련한 전\n문지식을 보유해야 하고, 내담자의 특성을 이해하려는 노력을 기울이며 상담 경험을\n축적하기 위한 노력을 지속해야 한다. 선수를 대상으로 한 심리기술훈련 과정에서 선수 개인의 문제나 경기력에 관련한\n문제를 주제로 상담을 진행하게 되는데 경기력은 단순히 경기력 관련 변수만으로 결\n정되고 영향을 미치는 것이 아니라 배경 변수로 개인의 환경이나 상황도 의미 있는\n변수로 영향을 미친다. 특히, 심리기술훈련에 참가하는 개인의 인적 문제는 스포츠와\n관련 없이 발생하는 문제이기는 하지만 심리기술훈련의 효과영향을 미치기 때문에\n심리기술훈련 단계에서 개인의 인적 문제를 조절하고 관리해야 한다.', '상담은 도움을 필요로 하는 대상인 내담자와 도움을 주는 상담자 사이에 형성된\n인간관계 및 도움을 주고받는 과정이다. 상담을 통해 내담자는 정보를 수집하고, 심\n리적 문제나 위기를 해결하는데 도움을 받을 뿐만 아니라 당면한 문제의 해결책을\n찾는데 도움을 받게 된다. 심리기술훈련의 방법으로서 상담은 선수의 인간적 측면에\n초점을 두고 개인적 성장을 도모하려는 일련의 개입과 상호작용 과정을 의미한다. 상담은 내담자의 필요에 의해 상담을 신청하는 순간 시작되어 문제의 해결 단서\n가 확보되었을 때 종결하게 된다. 상담의 신청 과정에서 우리나라의 문화적 특수성\n으로 인해 개인의 문제를 외부에 노출하기 꺼리는 우리나라의 문화적 특성으로 인 해 전화 상담이나 인터넷을 이용한 상담\n등 상담을 신청하는 방식을 다양화 시킬\n필요가 있다.', '가) 끌기 동작 중 일시정지 후 다시 끌어올리는 행위(연속동작행위 위반) 나) 발 이외 신체의 어느 부분이 경기대에 닿을 경우 다) 마무리 들기 동작에서 팔을 수평으로 혹은 완전히

['가) 끌기 동작 중 일시정지 후 다시 끌어올리는 행위(연속동작행위 위반) 나) 발 이외 신체의 어느 부분이 경기대에 닿을 경우 다) 마무리 들기 동작에서 팔을 수평으로 혹은 완전히 펴지 못할 경우\n라) 팔을 펴는 동안 순간적으로 정지 시\n마) 밀어내듯이 들기를 마무리하기\n바) 정상자세로 돌아오는 동안 팔꿈치를 구부리거나 펴는 경우\n사) 경기진행 중에 경기대를 벗어날 경우\n아) 바벨을 심판이 신호를 보내기 전에 경기대에 내려놓는 행위\n자) 심판 신호 후에 바벨을 떨어뜨리는 경우\n차) 양발과 바벨을 동일선상이 되지 못하거나, 몸통을 수평으로 평행이 되지 못할\n경우\n카) 바벨을 경기대에 내려놓지 못할 경우 (예：바벨 전체가 처음 경기대에 닿아야\n한다.',
 '인상이 취약한 한국선수들과 세계역도강국이며 인상기술이 뛰어난 중국선수들과\n의 전지훈련을 통하여 함께 훈련함으로써 직접 보고 느낀 점은 다음과 같다. 이 표는 한국 선수와 중국 선수의 인상 기술 차이점을 설명하고 있으며, 특히 허리의 S자형 유지의 중요성을 강조하고 있다. 허리가 S자형을 유지하지 못하면 허리 디스크에 압력이 가해져 부상의 위험이 증가하고, 힘을 효과적으로 발휘하기 어렵다는 점을 지적한다. 또한, 바벨과 어깨 사이의 길이가 짧아 팔로 바벨을 당기게 되어 몸통이 후방으로 빠지는 경향이 있으며, 이는 허리와 다리의 힘이 바벨에 전달되지 않는 문제를 야기한다. 스타트 이후 상체가 앞쪽으로 기울어지는 경향과 발의 유연성이 부족한 점도 언급되어 있다.',
 '잡아채기 동작은 끌기 동작(무릎 밑)이 끝나고 무릎을 지나면서 상방으로 신전된\n무릎이 이중 굽힘 동작을 하기 위하여 하방으로 내려간다. 폭발적으로 힘을 쓰는\n잡아채기 동작으로 앉아받기 전 단계로 정의한다. 1) 무릎을 지나면서 대퇴의 3/1, 또는 중간에서 고관절이 펴지는데 이 지점이 가장\n바벨을 들어 올리는데 유리한 단계라 할 수 있다. 2) 순간에 무릎이 바벨 아래로 이동하고, 그립 앞에 위치한다. 3) 팔은 곧게 펴고

일단 쉽게 생각하자.
1. intent 분류 정확성 테스트하자.
2. 복합질의 중에서 실현 가능한 것, 실현 불가능한 것을 선택해서 수행할 수 있도록 하자.
    * 실현 가능한 것은 정확하게 실현 되도록
    * 실현할 수 없는 것은 hitl을 이용해서 선택하도록 하자.